# 06 - Transfer, the CNN baseline, and the mismatch detector

The last three results, and the ones the thesis claim actually rests on.

- **the baseline** (§8.3). A CNN that regresses `(xc, yc, R)` straight from the receiver ring.
  It is 100x cheaper than the inversion and it is the honest comparison: if it matches the
  four-stage pipeline, the differentiable forward model bought nothing.
- **out-of-family transfer** (§8.5, §11.2 step 13). Ellipses and two-void geometries, solved
  with the FDTD, inverted with a surrogate that was trained on circles only. Nothing is
  retrained.
- **the mismatch detector** (§11.2 step 12). The final relative misfit as a test statistic for
  "is this defect the kind of defect I can represent?", scored as an ROC.

The out-of-family data comes from fresh solver runs, not from the network. That is the whole
point: the network has to be asked about a shape it has never seen, and the answer has to be
compared against the real physics of that shape.

**Runtime.** The FDTD solves are minutes; the CNN trains in minutes; the inversions dominate.
Set `QUICK = True` for a short pass. Headless:

```
modal run modal_app.py::transfer_and_detector
```

In [ ]:
# The repo root holds bootstrap.py; these notebooks live one level down in notebooks/.
# bootstrap.setup() puts the repo on sys.path, installs anything missing, picks the
# device, finds a persistent data directory, and turns TF32 off.  It is the only
# platform-aware code in this notebook.
import pathlib
import sys

_here = pathlib.Path.cwd()
_root = next((p for p in (_here, *_here.parents) if (p / "bootstrap.py").exists()), None)
assert _root is not None, "run this notebook from inside the fno-wave-inverse checkout"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import bootstrap

E = bootstrap.setup()
DEV = E.device

In [ ]:
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 9,
                     "figure.facecolor": "white", "axes.grid": True,
                     "grid.alpha": 0.25, "axes.axisbelow": True})

from src import config as cfg


def _np(v):
    # Anything printable/plottable, as a numpy array.  The library returns torch
    # tensors, numpy arrays and lists interchangeably depending on the entry point.
    if torch.is_tensor(v):
        return v.detach().cpu().numpy()
    return np.asarray(v)


def savefig(fig, name):
    p = E.figures / name
    fig.savefig(p, bbox_inches="tight")
    print("wrote", p)
    return p


def dump(obj, name):
    p = E.results / name
    p.write_text(json.dumps(obj, indent=1, default=float))
    print("wrote", p)
    return p


def table(rows, headers):
    w = [max(len(str(h)), *(len(f"{r[i]}") for r in rows)) if rows else len(str(h))
         for i, h in enumerate(headers)]
    line = "  ".join(f"{h:>{w[i]}}" for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))
    for r in rows:
        print("  ".join(f"{r[i]:>{w[i]}}" for i in range(len(headers))))

In [ ]:
# The three splits and the checkpoint the later notebooks read.  Nothing here writes.
paths = {k: E.datasets / f"{k}.h5" for k in ("train", "val", "test")}
for k, p in paths.items():
    print(f"{k:>6}  {'present' if p.exists() else 'MISSING':>7}  "
          f"{(p.stat().st_size / 1e9 if p.exists() else 0):6.2f} GB  {p}")

CKPT_DIR = E.checkpoints / "full"
CKPT = CKPT_DIR / "best.pt"
print(f"\nckpt   {'present' if CKPT.exists() else 'MISSING':>7}  {CKPT}")

In [ ]:
QUICK = True

N_IN = 12 if QUICK else 24            # in-family cases, the detector's null class
N_ELLIPSE = 6 if QUICK else 12        # out-of-family: ellipses
N_TWO = 6 if QUICK else 12            # out-of-family: two voids
N_SEED = 6 if QUICK else 12           # stage-0 seeding comparison
REG_EPOCHS = 30 if QUICK else 200
SNR = 30.0                            # the SNR every number in this notebook is quoted at

print(f"{'QUICK' if QUICK else 'FULL'}: {N_IN} in-family, "
      f"{N_ELLIPSE} ellipse + {N_TWO} two-void out-of-family, at {SNR:.0f} dB")
print(f"RingCNN: {REG_EPOCHS} epochs")

In [ ]:
import csv

import h5py

from src import training
from src.data import generate as G
from src.data.dataset import load_incident, load_inversion_case
from src.geometry.sdf import (Circle, Ellipse, TwoCircle, fine_coords,
                              geometry_channels, material_fields, soft_indicator)
from src.inverse import invert as INV
from src.inverse.misfit import InverseCase, Objective, SurrogateForward
from src.models import cnn_regressor as CNN
from src.solver import harmonic as H
from src.solver.fdtd_elastic import ElasticFDTD2D

for k in ("train", "val", "test"):
    assert paths[k].exists(), f"missing {paths[k]} -- run notebook 02"
assert CKPT.exists(), "no checkpoint -- run notebook 03"

model, meta = training.load(CKPT, device=DEV)
inc = load_incident(str(paths["test"]), device=DEV)
fwd = SurrogateForward(model, inc, device=DEV)
circle, ellipse, twocircle = Circle(), Ellipse(), TwoCircle()
om = H.omegas_tensor(DEV)

with h5py.File(paths["test"], "r") as f:
    src_test = f["samples/src_idx"][:]
    nu_test = f["samples/nu_idx"][:]
    theta_test = f["samples/theta"][:]
print(f"model {meta['arch']}, epoch {meta['epoch']}, device {DEV}")

## Part 1 -- the baseline that has to be beaten

A circular 1-D CNN over the 32 receivers, `4M + 3 = 83` input channels: real and imaginary
parts of both displacement components at all 20 frequencies, plus three broadcast conditioning
channels (source position and centred Poisson ratio). It regresses the **unconstrained** `z`,
not `theta`, because an MSE on `theta` would weight the two positions about 40x more heavily
than the radius purely through their units and the radius would never be learned.

The input is normalised by the **receiver-space** incident scale, not the domain scale. Those
differ by roughly two orders of magnitude -- the domain scale is set by the near-source
singularity -- and using the wrong one compresses the whole input to the bottom of float32's
useful range. Notebook 02 measured that ratio; this is the second place it matters.

Noise is added at the same `SNR` and in the same order (on the total A-scans, before the
incident field is subtracted) as the inversion sees, so the two are compared on the same data
and not on two different problems.

In [ ]:
gtr = torch.Generator().manual_seed(cfg.SEED)
gva = torch.Generator().manual_seed(cfg.SEED + 1)
gte = torch.Generator().manual_seed(cfg.SEED + 2)

t0 = time.perf_counter()
rtr = CNN.ring_features(str(paths["train"]), snr_db=SNR, generator=gtr, device=DEV)
rva = CNN.ring_features(str(paths["val"]), snr_db=SNR, generator=gva, device=DEV)
rte = CNN.ring_features(str(paths["test"]), snr_db=SNR, generator=gte, device=DEV)
print(f"ring features in {time.perf_counter()-t0:.1f} s")
print(f"train {tuple(rtr.x.shape)}  val {tuple(rva.x.shape)}  test {tuple(rte.x.shape)}")
print(f"RING_CHANNELS = {CNN.RING_CHANNELS} = 4 x {cfg.M_FREQ} + 3   "
      f"({rtr.x.numel()*4/1e6:.1f} MB in memory for train)")

In [ ]:
net, hist = CNN.train_regressor(rtr, rva, device=DEV, epochs=REG_EPOCHS,
                                log_every=max(REG_EPOCHS // 8, 1))
CNN.save(net, E.checkpoints / "ringcnn.pt")
print(f"\n{net.n_params():,} parameters "
      f"({net.n_params()/model.effective_params():.3%} of the FNO's)")

held_te = np.isin(src_test, cfg.SRC_HELDOUT)
sc_val = CNN.score(net, rva)
sc_te = CNN.score(net, rte)
sub = lambda m: CNN.RingData(rte.x[m], rte.theta[m], rte.nu[m], rte.src_idx[m])
m_he = torch.from_numpy(held_te).to(rte.x.device)
sc_tr_src = CNN.score(net, sub(~m_he))
sc_he_src = CNN.score(net, sub(m_he))

table([(k, f"{sc_val[k]}", f"{sc_te[k]}", f"{sc_tr_src[k]}", f"{sc_he_src[k]}")
       for k in ("position_ls_mean", "position_ls_median", "radius_ls_mean",
                 "success_rate")],
      ["RingCNN", "val", "test", "test: trained src", f"test: held out {cfg.SRC_HELDOUT}"])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.0))
ax[0].semilogy(hist["train"], lw=1.0, label="train")
ax[0].semilogy(hist["val"], lw=1.0, label="val")
ax[0].set(xlabel="epoch", ylabel="MSE on z", title="RingCNN training")
ax[0].legend(fontsize=8)

with torch.no_grad():
    th_hat = net.predict_theta(rte.x, rte.nu, circle)
lam_te = np.array([cfg.cs_over_cp(float(v)) / cfg.FC for v in _np(rte.nu)])
err_cnn = (_np((th_hat[:, :2] - rte.theta[:, :2]).pow(2).sum(-1).sqrt()) / lam_te)
bins = np.linspace(0, max(err_cnn.max() * 1.02, 0.5), 40)
ax[1].hist(err_cnn[~held_te], bins=bins, alpha=0.75, label="trained sources")
ax[1].hist(err_cnn[held_te], bins=bins, alpha=0.75, label=f"held out {cfg.SRC_HELDOUT}")
ax[1].axvline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0,
              label=f"gate {cfg.GATE_POSITION_LS} lambda_s")
ax[1].set(xlabel="position error / lambda_s", ylabel="count",
          title="RingCNN on the test split")
ax[1].legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "06_ringcnn.png")
plt.show()

### Does stage 0 actually help?

`invert` *adds* the CNN's guess to the screen's survivors rather than replacing them, and the
asymmetry is deliberate: if the CNN is right the extra candidate costs one row in a batch of 17,
and if the defect is out of distribution -- which is what the rest of this notebook is about --
the CNN's guess can be badly wrong and must not be the only starting point.

The comparison below is the same cases inverted with and without the seed. What is being looked
for is not a better final answer -- both should reach the same basin on in-family data -- but a
cheaper path to it, visible as a lower stage-2 objective from the first step.

In [ ]:
rng = np.random.default_rng(cfg.SEED + 7)
sel_seed = rng.choice(len(theta_test), size=N_SEED, replace=False)


def get_case(i, snr_db=SNR, seed_off=0):
    g = torch.Generator().manual_seed(int(cfg.SEED) + int(i) + int(seed_off))
    return InverseCase.from_dict(
        load_inversion_case(str(paths["test"]), int(i), snr_db=snr_db,
                            generator=g)).to(DEV)


rows, seed_rec = [], []
for i in sel_seed:
    case = get_case(int(i))
    j = int(nu_test[i])
    x = CNN.pack_ring(case.d_obs.cpu(), src_idx=torch.tensor([case.src_idx]),
                      nu=torch.tensor([case.nu]),
                      scale=inc["scale_recv"][case.src_idx, j].cpu().unsqueeze(0))
    with torch.no_grad():
        th0 = net.predict_theta(x.to(DEV), torch.tensor([case.nu], device=DEV),
                                circle)[0]
    r_no = INV.invert(fwd, case, family=circle)
    r_yes = INV.invert(fwd, case, family=circle, theta_init=th0)
    e0 = float((th0[:2].cpu() - case.theta_true[:2].cpu()).norm()) / case.lambda_s
    rows.append((int(i), f"{e0:.3f}", f"{r_no.position_error_ls:.4f}",
                 f"{r_yes.position_error_ls:.4f}",
                 f"{r_no.stages['stage2_trace'][0]:.3e}",
                 f"{r_yes.stages['stage2_trace'][0]:.3e}"))
    seed_rec.append(dict(index=int(i), stage0_error_ls=e0,
                         no_seed=r_no.position_error_ls,
                         seeded=r_yes.position_error_ls,
                         no_seed_misfit=r_no.misfit, seeded_misfit=r_yes.misfit))
table(rows, ["test i", "stage 0 err", "final, no seed", "final, seeded",
             "stage 2 J[0], no seed", "stage 2 J[0], seeded"])
d0 = np.array([r["no_seed"] for r in seed_rec])
d1 = np.array([r["seeded"] for r in seed_rec])
print(f"\nmedian position error  no seed {np.median(d0):.4f}   "
      f"seeded {np.median(d1):.4f} lambda_s")

## Part 2 -- out-of-family geometries, solved properly

Ellipses (aspect ratio 1.5-2.4, random orientation) and pairs of circles (separation 1.6-3.0
mean radii). Both families are in `sdf.py` and neither appears anywhere in the training data.

The solver run is set up exactly as `generate.py` sets it up -- same fine grid, same interface
width in *physical* units (`EPS_LEN_PHYS`, the width the network sees, not that many cells of
the finer grid), same source injection, same receiver sampling from the downsampled field. The
only change is which SDF makes `chi`. Anything else would confound "the network has not seen
this shape" with "the data was made differently".

Sources are drawn from `SRC_TRAIN` on purpose. The point of this experiment is to vary one
thing, and that thing is the shape.

**The equivalent circle.** Scoring a circle fit against a non-circular truth needs a reference.
The convention here is equal area: `R_eq = sqrt(a b)` for an ellipse, `sqrt(R1^2 + R2^2)` for
two voids, with the area-weighted centroid as the centre. It is a convention and not a ground
truth -- there is no correct circle for an ellipse -- which is exactly why the misfit, and not
the position error, is what the detector is built on.

In [ ]:
def sample_out(n, fam, rng):
    th, ss, jj = [], [], []
    while len(th) < n:
        j = int(rng.integers(len(cfg.NU_LIST)))
        lam = cfg.cs_over_cp(cfg.NU_LIST[j]) / cfg.FC
        s = int(rng.choice(cfg.SRC_TRAIN))
        sx, sy = cfg.SOURCE_XY[s]
        if fam.name == "ellipse":
            r_eq = float(rng.uniform(1.15 * cfg.R_MIN_LS, 0.85 * cfg.R_MAX_LS)) * lam
            ar = float(rng.uniform(1.5, 2.4))
            a, b = r_eq * math.sqrt(ar), r_eq / math.sqrt(ar)
            ext = a
        else:
            r1 = float(rng.uniform(cfg.R_MIN_LS, 0.8 * cfg.R_MAX_LS)) * lam
            r2 = r1 * float(rng.uniform(0.6, 1.0))
            # 1.6-3.0 mean radii apart: overlapping peanut at the low end,
            # two resolved voids at the high end.
            sep = 0.5 * (r1 + r2) * float(rng.uniform(1.6, 3.0))
            ang = float(rng.uniform(0, 2 * math.pi))
        pad = cfg.BOUNDARY_KEEPOUT_LS * lam
        if fam.name == "ellipse":
            keep = ext + pad
            if 2 * keep >= cfg.L_DOMAIN:
                continue
            xc = float(rng.uniform(keep, cfg.L_DOMAIN - keep))
            yc = float(rng.uniform(keep, cfg.L_DOMAIN - keep))
            if math.hypot(xc - sx, yc - sy) < ext + G.SRC_KEEPOUT_LS * lam:
                continue
            al = float(rng.uniform(-math.pi / 2, math.pi / 2))
            th.append([xc, yc, a, b, al])
        else:
            keep = max(r1, r2) + 0.5 * sep + pad
            if 2 * keep >= cfg.L_DOMAIN:
                continue
            xc = float(rng.uniform(keep, cfg.L_DOMAIN - keep))
            yc = float(rng.uniform(keep, cfg.L_DOMAIN - keep))
            dx_, dy_ = 0.5 * sep * math.cos(ang), 0.5 * sep * math.sin(ang)
            c1 = (xc + dx_, yc + dy_)
            c2 = (xc - dx_, yc - dy_)
            ok = all(math.hypot(c[0] - sx, c[1] - sy) > r + G.SRC_KEEPOUT_LS * lam
                     for c, r in ((c1, r1), (c2, r2)))
            ok &= all(pad + r <= v <= cfg.L_DOMAIN - pad - r
                      for c, r in ((c1, r1), (c2, r2)) for v in c)
            if not ok:
                continue
            th.append([c1[0], c1[1], r1, c2[0], c2[1], r2])
        ss.append(s)
        jj.append(j)
    return (np.asarray(th, np.float32), np.asarray(ss, np.int64),
            np.asarray(jj, np.int64))


def equivalent_circle(theta, fam):
    t = np.atleast_2d(np.asarray(theta, np.float64))
    if fam.name == "ellipse":
        return np.stack([t[:, 0], t[:, 1], np.sqrt(t[:, 2] * t[:, 3])], axis=-1)
    a1, a2 = t[:, 2] ** 2, t[:, 5] ** 2
    w = a1 + a2
    return np.stack([(a1 * t[:, 0] + a2 * t[:, 3]) / w,
                     (a1 * t[:, 1] + a2 * t[:, 4]) / w,
                     np.sqrt(w)], axis=-1)


rng = np.random.default_rng(cfg.SEED + 11)
th_e, src_e, nu_e = sample_out(N_ELLIPSE, ellipse, rng)
th_t, src_t, nu_t = sample_out(N_TWO, twocircle, rng)
print(f"ellipses  {th_e.shape}  aspect ratios "
      f"{np.round(th_e[:, 2]/th_e[:, 3], 2)}")
print(f"two-void  {th_t.shape}  radius ratios "
      f"{np.round(th_t[:, 5]/th_t[:, 2], 2)}")

In [ ]:
inc_ascans = inc["ascans"].cpu()
inc_scale_r = inc["scale_recv"].cpu()


def solve_out(theta, fam, src_idx, nu_idx, *, snr_db=SNR, seed=0):
    """FDTD -> scattered displacement phasors at the ring, [B,R,2,M] complex."""
    yy_f, xx_f = fine_coords(device=DEV)
    out = []
    for lo in range(0, len(theta), cfg.GEN_BATCH):
        hi = min(lo + cfg.GEN_BATCH, len(theta))
        th = torch.as_tensor(theta[lo:hi], device=DEV)
        chi = soft_indicator(fam.sdf(th, yy_f, xx_f), G.EPS_LEN_PHYS)
        lm = [cfg.lame_from_nu(cfg.NU_LIST[int(j)]) for j in nu_idx[lo:hi]]
        lam0 = torch.tensor([v[0] for v in lm], device=DEV).view(-1, 1, 1)
        mu0 = torch.tensor([v[1] for v in lm], device=DEV).view(-1, 1, 1)
        lam, mu, rho = material_fields(chi, lam0, mu0)
        sim = ElasticFDTD2D(lam, mu, rho)
        src = [cfg.net_to_fine(*cfg.SOURCES_NET[int(s)]) for s in src_idx[lo:hi]]
        res = sim.run(src, nt=cfg.NT, recv_yx=cfg.RECEIVERS_NET, omegas=om)
        a_tot = res.ascans.cpu()
        a_inc = inc_ascans[torch.as_tensor(src_idx[lo:hi]),
                           torch.as_tensor(nu_idx[lo:hi])]
        if snr_db is not None:
            g = torch.Generator().manual_seed(int(cfg.SEED) + seed + lo)
            a_tot = H.add_measurement_noise(a_tot, snr_db, generator=g)
        d = (H.displacement_from_ascans(a_tot, omegas=om)
             - H.displacement_from_ascans(a_inc, omegas=om))
        out.append(d)
        del sim, chi, lam, mu, rho
    if DEV.startswith("cuda"):
        torch.cuda.empty_cache()
    return torch.cat(out)


t0 = time.perf_counter()
d_e = solve_out(th_e, ellipse, src_e, nu_e, seed=100)
d_t = solve_out(th_t, twocircle, src_t, nu_t, seed=200)
print(f"{len(th_e)+len(th_t)} out-of-family FDTD solves in "
      f"{(time.perf_counter()-t0)/60:.1f} min")
print(f"d_ellipse {tuple(d_e.shape)}   d_two {tuple(d_t.shape)}")

eq_e = equivalent_circle(th_e, ellipse)
eq_t = equivalent_circle(th_t, twocircle)
cases_e = [InverseCase(d_obs=d_e[k:k+1], src_idx=int(src_e[k]),
                       nu_idx=int(nu_e[k]), snr_db=SNR,
                       theta_true=torch.tensor(eq_e[k], dtype=torch.float32)
                       ).to(DEV) for k in range(len(th_e))]
cases_t = [InverseCase(d_obs=d_t[k:k+1], src_idx=int(src_t[k]),
                       nu_idx=int(nu_t[k]), snr_db=SNR,
                       theta_true=torch.tensor(eq_t[k], dtype=torch.float32)
                       ).to(DEV) for k in range(len(th_t))]

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(11.6, 5.8))
for r_, (th_, fam, eq, ttl) in enumerate([(th_e, ellipse, eq_e, "ellipse"),
                                          (th_t, twocircle, eq_t, "two voids")]):
    for c_ in range(3):
        k = c_ % len(th_)
        _, chi = geometry_channels(torch.as_tensor(th_[k:k+1], device=DEV), fam)
        _, chi_eq = geometry_channels(torch.as_tensor(eq[k:k+1], dtype=torch.float32,
                                                      device=DEV), circle)
        gx = np.arange(cfg.N_NET) * cfg.DX_NET
        a_ = ax[r_, c_]
        a_.imshow(_np(chi)[0], origin="lower", cmap="Greys", vmin=0, vmax=1,
                  extent=[gx[0], gx[-1], gx[0], gx[-1]])
        a_.contour(gx, gx, _np(chi_eq)[0], levels=[0.5], colors="C3",
                   linewidths=1.2)
        sx, sy = cfg.SOURCE_XY[int((src_e if r_ == 0 else src_t)[k])]
        a_.plot(sx, sy, "C0*", ms=10)
        a_.set(title=f"{ttl} {k}", xticks=[], yticks=[])
        a_.grid(False)
    a_ = ax[r_, 3]
    dd = (d_e if r_ == 0 else d_t)[0]
    im = a_.imshow(_np(dd.abs()[:, 0, :]), origin="lower", aspect="auto",
                   extent=[cfg.FREQS[0], cfg.FREQS[-1], 0, cfg.N_RECV],
                   cmap="magma")
    a_.set(xlabel="f / f_c", ylabel="receiver",
           title=f"|d_obs| x-component\n{ttl} 0, {SNR:.0f} dB")
    a_.grid(False)
    fig.colorbar(im, ax=a_, fraction=0.046)
ax[0, 0].set_ylabel("red: equal-area circle")
fig.tight_layout()
savefig(fig, "06_out_of_family_shapes.png")
plt.show()

## Part 3 -- the detector

Every case is inverted with `Circle()`. The in-family cases are drawn from the test split at
the same SNR and restricted to the same source pool, so the null class differs from the
alternative in the shape and in nothing else.

The statistic is the **final relative misfit**. After convergence, a circle fitted to a
circle's data leaves surrogate error plus measurement noise; a circle fitted to an ellipse's
data leaves structured residual it has no parameter to absorb. The misfit is normalised by
`||d_obs||^2`, which is what lets a single threshold work across SNRs, source positions and
defect sizes -- an absolute residual would need one threshold per acquisition.

Why this matters more than the position error: a localisation tool that silently returns a
confident wrong answer on an unmodelled defect is worse than one that says it does not know.

In [ ]:
sel_in = rng.choice(np.where(np.isin(src_test, cfg.SRC_TRAIN))[0], size=N_IN,
                    replace=False)
cases_in = [get_case(int(i), seed_off=500) for i in sel_in]

t0 = time.perf_counter()
res_in = INV.run_many(fwd, cases_in, family=circle, progress=tqdm)
res_e = INV.run_many(fwd, cases_e, family=circle, progress=tqdm)
res_t = INV.run_many(fwd, cases_t, family=circle, progress=tqdm)
print(f"\n{len(res_in)+len(res_e)+len(res_t)} inversions in "
      f"{(time.perf_counter()-t0)/60:.1f} min")

mis_in = [r.misfit for r in res_in]
mis_e = [r.misfit for r in res_e]
mis_t = [r.misfit for r in res_t]
roc = INV.detector_roc(mis_in, mis_e + mis_t)
roc_e = INV.detector_roc(mis_in, mis_e)
roc_t = INV.detector_roc(mis_in, mis_t)

table([("circle (in family)", len(mis_in), f"{np.median(mis_in):.4e}",
        f"{INV.summarise(res_in)['position_ls_median']:.4f}", "-"),
       ("ellipse", len(mis_e), f"{np.median(mis_e):.4e}",
        f"{INV.summarise(res_e)['position_ls_median']:.4f}", f"{roc_e['auc']:.3f}"),
       ("two voids", len(mis_t), f"{np.median(mis_t):.4e}",
        f"{INV.summarise(res_t)['position_ls_median']:.4f}", f"{roc_t['auc']:.3f}")],
      ["family inverted as a circle", "n", "median final misfit",
       "median position err (l_s)", "AUC vs in-family"])
print(f"\ncombined AUC = {roc['auc']:.4f}   "
      f"median misfit {roc['median_in']:.3e} (in) vs {roc['median_out']:.3e} (out), "
      f"a factor of {roc['median_out']/max(roc['median_in'],1e-30):.1f}")

In [ ]:
tpr = np.asarray(roc["tpr"])
fpr = np.asarray(roc["fpr"])
thr = np.asarray(roc["threshold"])
k_best = int(np.argmax(tpr - fpr))

fig, ax = plt.subplots(1, 3, figsize=(11.6, 3.2))
ax[0].plot(fpr, tpr, "-", lw=1.4, label=f"all out-of-family, AUC {roc['auc']:.3f}")
ax[0].plot(roc_e["fpr"], roc_e["tpr"], "--", lw=1.0,
           label=f"ellipse only, AUC {roc_e['auc']:.3f}")
ax[0].plot(roc_t["fpr"], roc_t["tpr"], ":", lw=1.2,
           label=f"two voids only, AUC {roc_t['auc']:.3f}")
ax[0].plot([0, 1], [0, 1], c="0.6", lw=0.8)
ax[0].plot(fpr[k_best], tpr[k_best], "ko", ms=6,
           label=f"Youden: TPR {tpr[k_best]:.2f} at FPR {fpr[k_best]:.2f}")
ax[0].set(xlabel="false positive rate (in-family flagged)",
          ylabel="true positive rate (out-of-family flagged)",
          title="figure 5: model-mismatch detector", aspect="equal")
ax[0].legend(fontsize=7)

for i, (v, lab) in enumerate([(mis_in, "circle"), (mis_e, "ellipse"),
                              (mis_t, "two voids")]):
    j = np.full(len(v), i, float) + rng.normal(0, 0.06, len(v))
    ax[1].semilogy(j, v, "o", ms=5, alpha=0.7)
    ax[1].semilogy([i - 0.25, i + 0.25], [np.median(v)] * 2, "k-", lw=1.6)
ax[1].axhline(thr[k_best], ls="--", c="C3", lw=1.0,
              label=f"threshold {thr[k_best]:.3e}")
ax[1].set(xticks=[0, 1, 2], xticklabels=["circle", "ellipse", "two voids"],
          ylabel="final relative misfit", title="the statistic itself")
ax[1].legend(fontsize=7.5)

pos_in = [r.position_error_ls for r in res_in]
pos_out = [r.position_error_ls for r in res_e + res_t]
ax[2].loglog(mis_in, np.maximum(pos_in, 1e-4), "o", ms=5, alpha=0.75,
             label="in family")
ax[2].loglog(mis_e + mis_t, np.maximum(pos_out, 1e-4), "s", ms=5, alpha=0.75,
             label="out of family (vs equal-area circle)")
ax[2].axhline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0)
ax[2].axvline(thr[k_best], ls="--", c="0.4", lw=1.0)
ax[2].set(xlabel="final misfit", ylabel="position error / lambda_s",
          title="it still localises --\nit just knows it does not fit")
ax[2].legend(fontsize=7)
fig.tight_layout()
savefig(fig, "06_fig5_detector_roc.png")
plt.show()

## Part 4 -- the transfer, with no retraining

The surrogate was trained on circles. It is now asked about ellipses, and the *only* thing that
changes is which `ShapeFamily` builds the two geometry input channels. The weights are frozen,
the band is the same, the optimiser is the same.

Stage 1 is skipped, and not for convenience: `screen` and `misfit_map` build their candidate
grid with a three-column `torch.stack`, so the screen is structurally circle-only. The ellipse
run is seeded from the converged circle fit -- `(xc, yc, R) -> (xc, yc, a=R, b=R, alpha=0)`,
a circle expressed in ellipse coordinates -- and refined from there. This is the honest
version of the §8.5 claim: the transfer works because the network learned an operator on
`(phi_tilde, chi)` fields rather than a map on three numbers, and the evidence is that
releasing two extra degrees of freedom *lowers the misfit* on data the network has never seen
the shape of.

In [ ]:
res_tr, rows = [], []
for k, (ce, rc) in enumerate(zip(cases_e, res_e)):
    x0, y0, r0 = (float(v) for v in rc.theta[:3])
    seed = torch.tensor([x0, y0, r0, r0, 0.0], dtype=torch.float32, device=DEV)
    ce5 = InverseCase(d_obs=ce.d_obs, src_idx=ce.src_idx, nu_idx=ce.nu_idx,
                      snr_db=ce.snr_db,
                      theta_true=torch.tensor(th_e[k], dtype=torch.float32)
                      ).to(DEV)
    r = INV.invert(fwd, ce5, family=ellipse, theta_init=seed, skip_screen=True)
    res_tr.append(r)
    tt, th_hat = th_e[k], _np(r.theta)
    ar_true, ar_hat = tt[2] / tt[3], th_hat[2] / max(th_hat[3], 1e-9)
    da = abs(((th_hat[4] - tt[4] + math.pi / 2) % math.pi) - math.pi / 2)
    rows.append((k, f"{ar_true:.2f}", f"{ar_hat:.2f}",
                 f"{math.degrees(da):.1f}", f"{r.position_error_ls:.4f}",
                 f"{rc.misfit:.3e}", f"{r.misfit:.3e}",
                 f"{100*(1 - r.misfit/max(rc.misfit,1e-30)):+.0f}%"))
table(rows, ["case", "true a/b", "fitted a/b", "orientation err (deg)",
             "position err (l_s)", "misfit as circle", "misfit as ellipse",
             "change"])

drop = np.array([1 - r.misfit / max(c.misfit, 1e-30)
                 for r, c in zip(res_tr, res_e)])
pos_tr = np.array([r.position_error_ls for r in res_tr])
print(f"\nmedian misfit reduction from releasing (b, alpha): {np.median(drop):+.1%}")
print(f"median position error, ellipse family: {np.median(pos_tr):.4f} lambda_s "
      f"(gate {cfg.GATE_POSITION_LS})")
print(f"improved in {int((drop > 0).sum())}/{len(drop)} cases")

In [ ]:
n_show = min(3, len(res_tr))
fig, ax = plt.subplots(1, n_show + 1, figsize=(3.0 * (n_show + 1), 3.1))
gx = np.arange(cfg.N_NET) * cfg.DX_NET
for k in range(n_show):
    a_ = ax[k]
    _, chi_true = geometry_channels(torch.as_tensor(th_e[k:k+1], device=DEV), ellipse)
    _, chi_c = geometry_channels(res_e[k].theta.unsqueeze(0).to(DEV), circle)
    _, chi_t = geometry_channels(res_tr[k].theta.unsqueeze(0).to(DEV), ellipse)
    a_.imshow(_np(chi_true)[0], origin="lower", cmap="Greys", vmin=0, vmax=1,
              extent=[gx[0], gx[-1], gx[0], gx[-1]])
    a_.contour(gx, gx, _np(chi_c)[0], levels=[0.5], colors="C3", linewidths=1.3)
    a_.contour(gx, gx, _np(chi_t)[0], levels=[0.5], colors="C2", linewidths=1.3)
    sx, sy = cfg.SOURCE_XY[int(src_e[k])]
    a_.plot(sx, sy, "C0*", ms=10)
    a_.set(title=f"ellipse {k}: grey truth,\nred circle fit, green ellipse fit",
           xticks=[], yticks=[])
    a_.grid(False)

a_ = ax[n_show]
a_.semilogy([0] * len(res_e), [r.misfit for r in res_e], "o", ms=5, alpha=0.7)
a_.semilogy([1] * len(res_tr), [r.misfit for r in res_tr], "s", ms=5, alpha=0.7)
for rc, rt in zip(res_e, res_tr):
    a_.semilogy([0, 1], [rc.misfit, rt.misfit], "-", c="0.6", lw=0.7)
a_.semilogy([-0.2, 0.2], [np.median(mis_e)] * 2, "k-", lw=1.6)
a_.semilogy([0.8, 1.2], [np.median([r.misfit for r in res_tr])] * 2, "k-", lw=1.6)
a_.axhline(np.median(mis_in), ls="--", c="C3", lw=1.0, label="in-family median")
a_.set(xticks=[0, 1], xticklabels=["circle\nfamily", "ellipse\nfamily"],
       ylabel="final misfit", title="no retraining, two extra\ndegrees of freedom")
a_.legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "06_transfer_ellipse.png")
plt.show()

### The baseline on the same data

The RingCNN has three output numbers and no notion of an ellipse, so the most it can do is
report the equal-area circle. It is given exactly the data the inversion was given -- the same
phasors, normalised by the same receiver-space scale -- and scored against the same convention.

This is the comparison the method has to win, and the reason it wins is structural rather than
about capacity: the regressor learned a map from ring data to three numbers on a distribution
of circles, and an ellipse is off that distribution with no mechanism to notice. The inversion
carries a forward model, so it can be handed a different family and a different parameter count
at inference time, and it reports a misfit that says when it is out of its depth.

In [ ]:
def cnn_on(d, src_idx, nu_idx, theta_ref):
    s = torch.as_tensor(src_idx)
    j = torch.as_tensor(nu_idx)
    nu = torch.tensor([cfg.NU_LIST[int(k)] for k in j], dtype=torch.float32)
    x = CNN.pack_ring(d.cpu(), src_idx=s, nu=nu, scale=inc_scale_r[s, j])
    rd = CNN.RingData(x, torch.as_tensor(theta_ref, dtype=torch.float32), nu, s)
    return CNN.score(net, rd.to(DEV))


cnn_e = cnn_on(d_e, src_e, nu_e, eq_e)
cnn_t = cnn_on(d_t, src_t, nu_t, eq_t)
inv_e = INV.summarise(res_e)
inv_t = INV.summarise(res_t)

table([("circle test split", f"{sc_te['position_ls_median']:.4f}",
        f"{INV.summarise(res_in)['position_ls_median']:.4f}"),
       ("ellipse (vs equal-area circle)", f"{cnn_e['position_ls_median']:.4f}",
        f"{inv_e['position_ls_median']:.4f}"),
       ("two voids (vs equal-area circle)", f"{cnn_t['position_ls_median']:.4f}",
        f"{inv_t['position_ls_median']:.4f}")],
      ["median position error (lambda_s)", "RingCNN (stage 0)",
       "four-stage inversion"])
print(f"\nRingCNN: {net.n_params():,} parameters, one forward pass per case.")
print(f"Inversion: {int(np.median([r.n_forward for r in res_in]))} surrogate "
      f"evaluations, {np.median([r.seconds for r in res_in]):.1f} s per case.")
print("The baseline is cheap and it is not wrong -- it is just unable to say when it "
      "is.")

## Part 5 -- the thesis table

Every headline number, with the notebook that produced it and the gate it is measured against.
Read from the JSON records in `E.results`, so this cell reports what was actually run rather
than what was intended; anything missing shows as `-` instead of silently defaulting.

In [ ]:
def load_rec(name):
    p = E.results / name
    return json.loads(p.read_text()) if p.exists() else {}


r1 = load_rec("01_solver_validation.json")
r2 = load_rec("02_dataset_generation.json")
r3 = load_rec("03_train_fno.json")
r4 = load_rec("04_forward_eval.json")
r5 = load_rec("05_inversion.json")


def pick(d, *path, default=None):
    for k in path:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d


def fmt(v, spec=".4g"):
    return "-" if v is None else format(v, spec) if isinstance(v, float) else str(v)


ck = pick(r1, "checks", default={}) or {}


def check(substr, field="value"):
    for k, v in ck.items():
        if substr in k.lower():
            return v.get(field)
    return None


rows = [
    ("solver checks passed", f"{pick(r1,'n_pass',default='-')} of "
     f"{pick(r1,'n_total',default='-')}", "5 of 5", "01"),
    ("deconvolution amplification",
     fmt(pick(r1, "deconvolution", "worst_amplification")), "< 25", "01"),
    ("grid convergence, 256 vs 512", fmt(check("grid")),
     f"< {cfg.GATE_GRID_CONVERGENCE}", "01"),
    ("PML residual", fmt(check("pml"), ".2e"),
     f"< {cfg.GATE_PML_RESIDUAL}", "01"),
    ("dataset re-solve spot check", fmt(pick(r2, "spot_check", "max"), ".2e"),
     f"< {pick(r2,'spot_check','gate',default='-')}", "02"),
    ("wrap-around tail energy",
     fmt(pick(r2, "tail_energy_fraction", "max"), ".2e"), "< 1e-3", "02"),
    ("surrogate field rel-L2 (test)", fmt(pick(r4, "test", "rel_l2")),
     f"< {cfg.GATE_REL_L2}", "04"),
    ("surrogate arrival error", fmt(pick(r4, "test", "phase")),
     f"< {cfg.GATE_ARRIVAL_PERIODS} periods", "04"),
    ("held-out-source penalty",
     fmt(pick(r4, "per_sample", "heldout_penalty")), "reported", "04"),
    ("physics-loss ablation, rel-L2",
     f"{fmt(pick(r3,'arms','nophys','rel_l2'))} -> "
     f"{fmt(pick(r3,'arms','full','rel_l2'))}", "physics <= none", "03"),
    ("L_phys floor (true field)", fmt(pick(r3, "phys_floor_true_field"), ".4e"),
     "reported", "03"),
    ("gradient check, significant figures",
     fmt(pick(r5, "gradient_check", "digits"), ".2f"),
     f">= {cfg.GATE_GRAD_SIGFIGS}", "05"),
    ("inversion success rate at 30 dB",
     fmt(pick(r5, "step11_30db", "all", "success_rate"), ".1%"),
     f">= {cfg.GATE_SUCCESS_RATE:.0%}", "05"),
    ("  on held-out illuminations",
     fmt(pick(r5, "step11_30db", "heldout", "success_rate"), ".1%"),
     f">= {cfg.GATE_SUCCESS_RATE:.0%}", "05"),
    ("basin width along/across",
     f"{fmt(pick(r5,'basin','full','along_ls'),'.3f')} / "
     f"{fmt(pick(r5,'basin','full','across_ls'),'.3f')} lambda_s",
     "~0.25, elongated", "05"),
    ("RingCNN baseline, median position",
     f"{sc_te['position_ls_median']:.4f} lambda_s", "for comparison", "06"),
    ("inversion, median position (in family)",
     f"{INV.summarise(res_in)['position_ls_median']:.4f} lambda_s",
     f"< {cfg.GATE_POSITION_LS}", "06"),
    ("mismatch detector AUC", f"{roc['auc']:.4f}", "> 0.9 desirable", "06"),
    ("ellipse transfer, median position",
     f"{np.median(pos_tr):.4f} lambda_s", f"< {cfg.GATE_POSITION_LS}", "06"),
    ("ellipse transfer, misfit change",
     f"{np.median(drop):+.1%}", "negative is a failure", "06"),
]
table(rows, ["quantity", "value", "gate / expectation", "notebook"])

In [ ]:
csv_path = E.results / "thesis_table.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["quantity", "value", "gate", "notebook"])
    w.writerows(rows)
print(f"wrote {csv_path}")

record = {
    "device": DEV, "gpu": E.gpu_name, "quick": QUICK, "snr_db": SNR,
    "ringcnn": dict(params=net.n_params(), epochs=REG_EPOCHS,
                    val=sc_val, test=sc_te, test_trained_src=sc_tr_src,
                    test_heldout_src=sc_he_src,
                    on_ellipse=cnn_e, on_two_circle=cnn_t,
                    history=hist),
    "stage0_seeding": seed_rec,
    "out_of_family": dict(
        ellipse=dict(theta=th_e.tolist(), src=src_e.tolist(), nu=nu_e.tolist(),
                     equivalent_circle=eq_e.tolist(), misfit=mis_e,
                     summary=inv_e),
        two_circle=dict(theta=th_t.tolist(), src=src_t.tolist(), nu=nu_t.tolist(),
                        equivalent_circle=eq_t.tolist(), misfit=mis_t,
                        summary=inv_t)),
    "in_family": dict(indices=sel_in.tolist(), misfit=mis_in,
                      summary=INV.summarise(res_in)),
    "detector": dict(auc=roc["auc"], auc_ellipse=roc_e["auc"],
                     auc_two_circle=roc_t["auc"],
                     median_in=roc["median_in"], median_out=roc["median_out"],
                     youden_threshold=float(thr[k_best]),
                     youden_tpr=float(tpr[k_best]), youden_fpr=float(fpr[k_best]),
                     tpr=roc["tpr"], fpr=roc["fpr"],
                     threshold=roc["threshold"]),
    "transfer_ellipse": dict(
        theta=[_np(r.theta).tolist() for r in res_tr],
        misfit_as_circle=mis_e, misfit_as_ellipse=[r.misfit for r in res_tr],
        misfit_reduction=drop.tolist(), position_ls=pos_tr.tolist(),
        median_reduction=float(np.median(drop)),
        median_position_ls=float(np.median(pos_tr))),
    "thesis_table": [list(r) for r in rows],
}
dump(record, "06_transfer_and_detector.json")

## The never-cut checklist

§11.3, in order. These are the things that stay in even when time runs out, because each one is
the only place a particular kind of silent wrongness can be caught. A `FAIL` or a `-` below is
not a note in the discussion section; it is a number that should not be quoted.

In [ ]:
def ok(v, gate, cmp="lt"):
    if v is None:
        return None
    return (v < gate) if cmp == "lt" else (v >= gate)


checks = [
    ("solver sanity checks 1-5",
     (pick(r1, "n_pass") == pick(r1, "n_total") and pick(r1, "include_slow"))
     if r1 else None,
     "an unvalidated solver makes every downstream metric a report on the wrong "
     "physics"),
    ("dataset re-solve spot check", pick(r2, "spot_check", "passed"),
     "the file on disk is what the solver produced"),
    ("forward rel-L2 and arrival gates",
     all(pick(r4, "gates", default={}).values()) if pick(r4, "gates") else None,
     "the surrogate is the operator being inverted; its error is a floor"),
    ("held-out-source generalisation reported",
     pick(r4, "per_sample", "heldout_penalty") is not None,
     "the honest split -- SRC_HELDOUT appears in test and nowhere else"),
    ("gradient check to 3 significant figures",
     pick(r5, "gradient_check", "passed"),
     "a wrong gradient still converges, to the wrong answer"),
    ("misfit landscape figure",
     pick(r5, "basin", "full", "along_ls") is not None,
     "the basin width and its elongation are the resolution claim"),
    ("success rate at 30 dB",
     pick(r5, "step11_30db", "all", "gate_pass"),
     "the headline inverse result"),
    ("SNR sweep", bool(pick(r5, "snr_sweep")),
     "says whether 30 dB is inside the working range or on a cliff edge"),
    ("out-of-family transfer, no retraining",
     bool(np.median(drop) > 0), "the §8.5 claim, and the reason for a forward model"),
    ("model-mismatch detector",
     bool(roc["auc"] > 0.9),
     "a confident wrong answer on an unmodelled defect is the worst failure mode"),
    ("CNN baseline for comparison", bool(sc_te), "otherwise there is no claim"),
]
for name, v, why in checks:
    mark = "----" if v is None else ("PASS" if v else "FAIL")
    print(f"  {mark}  {name}")
    print(f"        {why}")

n_ok = sum(1 for _, v, _ in checks if v)
print(f"\n{n_ok} / {len(checks)}")
if n_ok == len(checks):
    print("\nEvery gate in §11.3 is met.  The six notebooks, their figures in "
          f"{E.figures}\nand their records in {E.results} are the complete "
          "experimental record.")
else:
    print("\nSome gates are unmet or unrun.  The list above says which notebook "
          "owns each one;\nrun that notebook rather than quoting around the gap.")

## What is left out, on purpose

- **No mixed precision anywhere.** The spectral weights are complex and the gradient check of
  notebook 05 needs double precision; TF32 is off for the same reason. The speedup was not
  worth an unexplained loss of three digits.
- **The screen is circle-only.** `screen` and `misfit_map` stack three columns, so an
  out-of-family inversion has to be seeded. That is a real limitation and it is stated rather
  than papered over -- extending the screen to five parameters would be a `16^5` grid, which is
  the actual reason it was not done.
- **`Ellipse` and `TwoCircle` are never trained on.** They exist to be transferred to. Adding
  them to the training distribution would make the §8.5 result vacuous.
- **The equal-area circle is a convention.** There is no correct circle for an ellipse, which
  is why the detector is built on the misfit and not on the position error.